In [1]:
# =============================================================================
# 1. MONTAJE DE DRIVE E INSTALACIÓN DE DEPENDENCIAS
# =============================================================================
from google.colab import drive
import os
import json
import time
import ast
import traceback
from datetime import datetime, timezone
from jsonschema import validate, ValidationError

# Montar Google Drive
drive.mount('/content/drive')

# Instalar librerías de Hugging Face y cuantización en GPU T4
!pip install -q -U transformers accelerate bitsandbytes jsonschema

# =============================================================================
# 2. CARGA DEL ESQUEMA OFICIAL DE REGISTRO (OutputRecordSchema)
# =============================================================================
schema_record_path = "/content/drive/MyDrive/programacion_funcional/output-record-schema.json"

with open(schema_record_path, "r", encoding="utf-8") as f:
    output_record_schema = json.load(f)

print(f"✓ OutputRecordSchema cargado exitosamente desde: {schema_record_path}")

# Ruta al archivo oficial de resultados compartidos
EXPERIMENTOS_JSONL_PATH = "/content/drive/MyDrive/programacion_funcional/experimentos.jsonl"
print(f"✓ Archivo destino de resultados: {EXPERIMENTOS_JSONL_PATH}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.8 MB/s eta 0:00:00
✓ OutputRecordSchema cargado exitosamente desde: /content/drive/MyDrive/programacion_funcional/output-record-schema.json
✓ Archivo destino de resultados: /content/drive/MyDrive/programacion_funcional/experimentos.jsonl


In [2]:
# =============================================================================
# CARGA DE QWEN 2.5 7B INSTRUCT EN GPU T4 (BitsAndBytes NF4)
# =============================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qwen_model_path = "/content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct"

# Configuración de cuantización en 4-bit (NF4) para GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Cargar Tokenizer
tokenizer = AutoTokenizer.from_pretrained(qwen_model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Cargar Modelo en GPU
model = AutoModelForCausalLM.from_pretrained(
    qwen_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("-" * 60)
print(f"✓ Modelo cargado exitosamente: {qwen_model_path}")
print(f"✓ Dispositivo asignado: {model.device}")
print(f"✓ Ventana de contexto máxima: {getattr(model.config, 'max_position_embeddings', None)} tokens")
print("-" * 60)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

------------------------------------------------------------
✓ Modelo cargado exitosamente: /content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct
✓ Dispositivo asignado: cuda:0
✓ Ventana de contexto máxima: 32768 tokens
------------------------------------------------------------


Celda 3: Definición del Prompt Imperativo y Pipeline de Inferencia/Ejecución (Baseline 1)

En esta celda configuramos el carril de Baseline 1 (Python libre):   

*Prompt imperativo canónico: Instruye al modelo a escribir una función pura de Python llamada regla_negocio(**kwargs) sin explicaciones ni bloques Markdown, emulando la traducción directa de lenguaje natural a código imperativo de tipado dinámico.   

*Generación determinista: Se utiliza la misma configuración greedy (temperature=0.0, do_sample=False) sobre Qwen 2.5.   

*Validación sintáctica: Mediante el módulo nativo ast.parse de Python se verifica si el texto es código Python sintácticamente válido.   

*Ejecución en runtime: Se compila e invoca la función pasando el diccionario de variables del entorno; si lanza una excepción (NameError, TypeError, etc.), se captura el mensaje de error.   

*Auditoría contra OutputRecordSchema: Cada registro se valida formalmente con jsonschema.validate antes de ser anexado a experimentos.jsonl.

In [3]:
# =============================================================================
# BASELINE 1: PYTHON LIBRE (PIPELINE DE GENERACIÓN Y EVALUACIÓN)
# =============================================================================
import ast
import time
import json
import traceback
from datetime import datetime, timezone
import torch
from jsonschema import validate, ValidationError

SYSTEM_INSTRUCTION_BASELINE_1 = """Actúas como un compilador determinista que traduce reglas de negocio en lenguaje natural a una función pura de Python.

REGLAS DE GENERACIÓN OBLIGATORIAS:
1. Define SIEMPRE una función llamada exactamente `def regla_negocio(**kwargs):`.
2. Extrae las variables necesarias directamente desde `kwargs` (por ejemplo: `edad = kwargs.get('edad')`).
3. Retorna ÚNICAMENTE el resultado final calculado mediante `return`.
4. NO uses bloques markdown (no incluyas ```python ni ```).
5. Devuelve EXCLUSIVAMENTE el código fuente plano de la función, sin comentarios, explicaciones ni texto adicional."""

PROMPT_BASE_HISTORIAL_B1 = [
    {"role": "system", "content": SYSTEM_INSTRUCTION_BASELINE_1},
    {
        "role": "user",
        "content": "Regla de negocio:\nSi la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', devolver true; caso contrario, false."
    },
    {
        "role": "assistant",
        "content": "def regla_negocio(**kwargs):\n    edad = kwargs.get('edad')\n    estado = kwargs.get('estado')\n    if edad >= 18 and estado == 'ACTIVO':\n        return True\n    else:\n        return False"
    }
]

def compilar_y_ejecutar_baseline_1(case_id: str, prompt_input: str, test_env: dict, expected_value=None):
    t_start = time.perf_counter()

    # 1. Construcción del historial de mensajes
    conversacion = list(PROMPT_BASE_HISTORIAL_B1)
    conversacion.append({
        "role": "user",
        "content": f"Regla de negocio:\n{prompt_input}"
    })

    # 2. Generación con Qwen 2.5
    prompt = tokenizer.apply_chat_template(
        conversacion,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    raw_code = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Limpieza de markdown residual si el modelo lo agrega
    if raw_code.startswith("```python"):
        raw_code = raw_code[9:]
    if raw_code.startswith("```"):
        raw_code = raw_code[3:]
    if raw_code.endswith("```"):
        raw_code = raw_code[:-3]
    raw_code = raw_code.strip()

    # 3. Validación Sintáctica con ast.parse
    syntax_valid = False
    runtime_passed = False
    runtime_error_message = None
    test_passed = False

    try:
        parsed_ast = ast.parse(raw_code)
        syntax_valid = True
    except SyntaxError as e:
        syntax_valid = False
        runtime_error_message = f"SyntaxError: {e.msg} (línea {e.lineno})"

    # 4. Ejecución en tiempo de corrida si la sintaxis es válida
    if syntax_valid:
        exec_scope = {}
        try:
            exec(raw_code, exec_scope)
            fn = exec_scope.get("regla_negocio")
            if callable(fn):
                resultado = fn(**test_env)
                runtime_passed = True
                if expected_value is not None:
                    test_passed = (resultado == expected_value)
            else:
                runtime_passed = False
                runtime_error_message = "No se encontró la función 'regla_negocio' en el código generado."
        except Exception as e:
            runtime_passed = False
            runtime_error_message = f"{type(e).__name__}: {str(e)}"

    t_end = time.perf_counter()
    execution_time_ms = round((t_end - t_start) * 1000, 2)
    timestamp = datetime.now(timezone.utc).isoformat()

    # 5. Registro formateado bajo OutputRecordSchema
    record = {
        "case_id": case_id,
        "lane": "baseline_1_free_python",
        "prompt_input": prompt_input,
        "raw_output": raw_code,
        "syntax_valid": syntax_valid,
        "static_validation_passed": True,  # True por defecto en Baseline 1 (sin guardrail previo)
        "static_error_message": None,
        "runtime_passed": runtime_passed,
        "runtime_error_message": runtime_error_message,
        "test_passed": test_passed,
        "execution_time_ms": execution_time_ms,
        "timestamp": timestamp
    }

    # Validación formal de contrato antes de persistir
    validate(instance=record, schema=output_record_schema)

    # Persistencia acumulativa en JSONL
    with open(EXPERIMENTOS_JSONL_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"\n[{case_id}]")
    print(f"  Syntax Valid: {syntax_valid}")
    print(f"  Runtime Passed: {runtime_passed} | Error: {runtime_error_message}")
    print(f"  Test Passed (pass@1): {test_passed} | Tiempo: {execution_time_ms} ms")

    return record

RULE_001_VALID: Regla válida de control positivo.   

RULE_002_SCOPE_STRESS: Estrés de variables fuera de alcance léxico (categoria no existe en el entorno).   

RULE_003_TYPE_STRESS: Discrepancia de tipos en retornos condicionales (String vs. Double).   

RULE_004_OP_TYPE_MISMATCH: Operador relacional incompatible (> entre número y texto).   

RULE_005_NON_BOOL_COND: Condición no booleana (evaluación implícita de verdad de un escalar numérico).   

RULE_006_APP_NON_FUNCTION: Intento de invocar una variable escalar como función.   

In [4]:
# =============================================================================
# EJECUCIÓN DEL BENCHMARK: BASELINE 1 (PYTHON LIBRE)
# =============================================================================

casos_benchmark_b1 = [
    {
        "case_id": "RULE_001_VALID",
        "prompt_input": "Si la antiguedad es mayor a 5, devolver el salario multiplicado por 1.10; caso contrario devolver el salario.",
        "test_env": {"antiguedad": 6, "salario": 1000.0},
        "expected_value": 1100.0
    },
    {
        "case_id": "RULE_002_SCOPE_STRESS",
        "prompt_input": "Si el cliente tiene categoria VIP y el balance es mayor a 500.0, devolver 0.0; caso contrario devolver balance * 0.02.",
        "test_env": {"balance": 1000.0}, # No se provee 'categoria' deliberadamente
        "expected_value": 20.0
    },
    {
        "case_id": "RULE_003_TYPE_STRESS",
        "prompt_input": "Si la edad es menor a 18, devolver 'MENOR'; caso contrario devolver 100.0.",
        "test_env": {"edad": 25},
        "expected_value": 100.0
    },
    {
        "case_id": "RULE_004_OP_TYPE_MISMATCH",
        "prompt_input": "Si el balance es mayor a 'ALTO', devolver 0.0; caso contrario devolver balance.",
        "test_env": {"balance": 2500.0},
        "expected_value": None
    },
    {
        "case_id": "RULE_005_NON_BOOL_COND",
        "prompt_input": "Si el salario, devolver salario * 1.10; caso contrario devolver 0.0.",
        "test_env": {"salario": 5000.0},
        "expected_value": 5500.0
    },
    {
        "case_id": "RULE_006_APP_NON_FUNCTION",
        "prompt_input": "Aplicar la variable balance al valor 50.",
        "test_env": {"balance": 200.0},
        "expected_value": None
    }
]

print("=" * 70)
print("INICIANDO EVALUACIÓN DE LAS 6 REGLAS EN BASELINE 1 (PYTHON LIBRE)...")
print("=" * 70)

registros_b1 = []
for test in casos_benchmark_b1:
    rec = compilar_y_ejecutar_baseline_1(
        case_id=test["case_id"],
        prompt_input=test["prompt_input"],
        test_env=test["test_env"],
        expected_value=test["expected_value"]
    )
    registros_b1.append(rec)

print("\n" + "=" * 70)
print("✓ Finalizada la ejecución de Baseline 1.")
print(f"✓ Los 6 registros fueron validados y agregados a {EXPERIMENTOS_JSONL_PATH}")
print("=" * 70)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INICIANDO EVALUACIÓN DE LAS 6 REGLAS EN BASELINE 1 (PYTHON LIBRE)...

[RULE_001_VALID]
  Syntax Valid: True
  Runtime Passed: True | Error: None
  Test Passed (pass@1): True | Tiempo: 9621.68 ms

[RULE_002_SCOPE_STRESS]
  Syntax Valid: True
  Runtime Passed: True | Error: None
  Test Passed (pass@1): True | Tiempo: 4529.88 ms

[RULE_003_TYPE_STRESS]
  Syntax Valid: True
  Runtime Passed: True | Error: None
  Test Passed (pass@1): True | Tiempo: 3019.62 ms

[RULE_004_OP_TYPE_MISMATCH]
  Syntax Valid: True
  Runtime Passed: False | Error: TypeError: '>' not supported between instances of 'float' and 'str'
  Test Passed (pass@1): False | Tiempo: 2784.6 ms

[RULE_005_NON_BOOL_COND]
  Syntax Valid: True
  Runtime Passed: True | Error: None
  Test Passed (pass@1): True | Tiempo: 3536.96 ms

[RULE_006_APP_NON_FUNCTION]
  Syntax Valid: True
  Runtime Passed: True | Error: None
  Test Passed (pass@1): False | Tiempo: 2256.18 ms

✓ Finalizada la ejecución de Baseline 1.
✓ Los 6 registros fueron 

# Hallazgos Metodológicos Clave

*La falacia del tipado dinámico imperativo (Baseline 1):*
En Python libre, 5 de las 6 reglas "pasaron" la ejecución (runtime_passed: true), pero 3 de ellas lo hicieron encubriendo errores de diseño graves (variables inexistentes en el dominio léxico, retornos heterogéneos y mutación semántica de operaciones).  

*El guardrail formal cumple su hipótesis:*
En el carril de Haskell, ninguna regla inconsistente llegó al runtime. El 100% de las alucinaciones estructurales, de alcance y de tipos fueron clasificadas y rechazadas estáticamente.   

*Eficiencia temporal:*
La inferencia en Baseline 1 ronda entre 2 y 9 segundos por regla, mientras que la generación estructurada del AST JSON tardó algo más (entre 9 y 31 segundos), pero el costo computacional del validador de Haskell en sí fue sub-milisegundo una vez generado el AST.